# CardioIA — Fase 4 · Parte 2: CNN, Transfer Learning e Gradio

> ⚠️ **AVISO CLÍNICO:** Este sistema é um protótipo acadêmico e **NÃO** substitui avaliação médica.  

Compara duas abordagens de classificação das imagens de ECG pré-processadas:
1. **CNN simples treinada do zero**
2. **Transfer Learning com VGG16** (ImageNet)

Ao final, um protótipo **Gradio** permite enviar um ECG e ver a classe prevista com probabilidades.

## 0. Ambiente e seeds

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)

EM_COLAB = False
try:
    import google.colab  # noqa: F401
    EM_COLAB = True
except ImportError:
    pass

if EM_COLAB:
    %pip install -q tensorflow opencv-python-headless scikit-learn matplotlib seaborn gradio pandas tqdm
    REPO = Path('/content/cardioAI')
    if not REPO.exists():
        REPO = Path('/content')
    FASE4 = REPO / 'fase4' if (REPO / 'fase4').exists() else REPO
else:
    cand = Path.cwd().resolve()
    FASE4 = None
    for p in [cand, *cand.parents]:
        if (p / 'fase4' / 'src' / 'pipeline_imagens.py').exists():
            FASE4 = p / 'fase4'
            break
        if (p / 'src' / 'pipeline_imagens.py').exists() and p.name == 'fase4':
            FASE4 = p
            break
    if FASE4 is None:
        raise FileNotFoundError('Não achei fase4/src/pipeline_imagens.py')

sys.path.insert(0, str(FASE4 / 'src'))

import numpy as np
np.random.seed(SEED)

import tensorflow as tf
tf.random.set_seed(SEED)

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from pipeline_imagens import CLASSES, gerar_geradores_keras, materializar_processado, ConfigPipeline

ORIGEM = FASE4 / 'data' / 'imagens_ecg_completo'
PROCESSADO = FASE4 / 'data' / 'processado'
MODELS = FASE4 / 'models'
EVIDENCIAS = FASE4 / 'docs' / 'evidencias'
MODELS.mkdir(parents=True, exist_ok=True)
EVIDENCIAS.mkdir(parents=True, exist_ok=True)

BATCH = 16
EPOCHS = 25
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
print('FASE4 =', FASE4)

## 1. Garantir dados processados e carregar geradores

In [ ]:
if not (PROCESSADO / 'manifesto_split.csv').exists():
    if not ORIGEM.exists() or sum(1 for _ in ORIGEM.rglob('*.jpg')) < 900:
        import importlib.util
        script = FASE4 / 'data' / 'baixar_ecg_mendeley.py'
        spec = importlib.util.spec_from_file_location('baixar_ecg_mendeley', script)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        mod.baixar_dataset(ORIGEM)
    materializar_processado(ConfigPipeline(origem=ORIGEM, destino=PROCESSADO))

fluxo_treino, fluxo_val, fluxo_teste = gerar_geradores_keras(
    PROCESSADO, batch_size=BATCH, seed=SEED, aumentar_treino=True
)
print('classes:', fluxo_treino.class_indices)
print('treino/val/teste:', fluxo_treino.samples, fluxo_val.samples, fluxo_teste.samples)

## 2. CNN simples do zero

In [ ]:
def construir_cnn_simples(n_classes: int = 4) -> keras.Model:
    modelo = keras.Sequential([
        layers.Input(shape=(224, 224, 3)),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Dropout(0.35),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(n_classes, activation='softmax'),
    ], name='cnn_simples')
    modelo.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return modelo

cnn = construir_cnn_simples()
cnn.summary()

In [ ]:
callbacks_cnn = [
    EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint(str(MODELS / 'cnn_simples.keras'), monitor='val_accuracy', save_best_only=True),
]

hist_cnn = cnn.fit(
    fluxo_treino,
    validation_data=fluxo_val,
    epochs=EPOCHS,
    callbacks=callbacks_cnn,
)
cnn.save(MODELS / 'cnn_simples_final.keras')

## 3. Transfer Learning — VGG16

In [ ]:
def construir_vgg16_tl(n_classes: int = 4) -> keras.Model:
    base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base.trainable = False
    entradas = keras.Input(shape=(224, 224, 3))
    # VGG16 no Keras Applications recomenda preprocess_input (caffe-style).
    # Como nossos geradores já estão em [0,1], reescalamos para [0,255] antes.
    x = layers.Rescaling(255.0)(entradas)
    x = keras.applications.vgg16.preprocess_input(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(128, activation='relu')(x)
    saidas = layers.Dense(n_classes, activation='softmax')(x)
    modelo = keras.Model(entradas, saidas, name='vgg16_tl')
    modelo.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return modelo, base

vgg, base_vgg = construir_vgg16_tl()
vgg.summary()

In [ ]:
# Geradores SEM rescale interno conflitante para VGG: já usamos [0,1] + Rescaling(255) no modelo.
callbacks_vgg = [
    EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint(str(MODELS / 'vgg16_tl.keras'), monitor='val_accuracy', save_best_only=True),
]

hist_vgg = vgg.fit(
    fluxo_treino,
    validation_data=fluxo_val,
    epochs=EPOCHS,
    callbacks=callbacks_vgg,
)

In [ ]:
# Fine-tune leve das últimas camadas convolucionais
base_vgg.trainable = True
for camada in base_vgg.layers[:-4]:
    camada.trainable = False

vgg.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

hist_vgg_ft = vgg.fit(
    fluxo_treino,
    validation_data=fluxo_val,
    epochs=10,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
        ModelCheckpoint(str(MODELS / 'vgg16_tl.keras'), monitor='val_accuracy', save_best_only=True),
    ],
)
vgg.save(MODELS / 'vgg16_tl_final.keras')

## 4. Avaliação no conjunto de teste

In [ ]:
def avaliar(modelo, fluxo, nome: str):
    fluxo.reset()
    probs = modelo.predict(fluxo, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    y_true = fluxo.classes
    labels = list(CLASSES)

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred)
    relatorio = classification_report(
        y_true, y_pred, target_names=labels, zero_division=0
    )

    print(f'=== {nome} ===')
    print(f'Acurácia={acc:.3f} | Precisão_macro={prec:.3f} | Recall_macro={rec:.3f} | F1_macro={f1:.3f}')
    print(relatorio)

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel('Previsto')
    ax.set_ylabel('Real')
    ax.set_title(f'Matriz de confusão — {nome}')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    fig_path = EVIDENCIAS / f'cm_{nome.lower().replace(" ", "_")}.png'
    fig.savefig(fig_path, dpi=140)
    plt.show()
    print('Salvo:', fig_path)

    return {
        'modelo': nome,
        'acuracia': float(acc),
        'precisao_macro': float(prec),
        'recall_macro': float(rec),
        'f1_macro': float(f1),
        'matriz_confusao': cm.tolist(),
        'relatorio': relatorio,
    }

# Recarrega melhores checkpoints se existirem
cnn_eval = keras.models.load_model(MODELS / 'cnn_simples.keras') if (MODELS / 'cnn_simples.keras').exists() else cnn
vgg_eval = keras.models.load_model(MODELS / 'vgg16_tl.keras') if (MODELS / 'vgg16_tl.keras').exists() else vgg

metricas_cnn = avaliar(cnn_eval, fluxo_teste, 'CNN_simples')
metricas_vgg = avaliar(vgg_eval, fluxo_teste, 'VGG16_TL')

In [ ]:
comparativo = pd.DataFrame([
    {k: metricas_cnn[k] for k in ['modelo', 'acuracia', 'precisao_macro', 'recall_macro', 'f1_macro']},
    {k: metricas_vgg[k] for k in ['modelo', 'acuracia', 'precisao_macro', 'recall_macro', 'f1_macro']},
])
display(comparativo)

melhor_nome = comparativo.loc[comparativo['f1_macro'].idxmax(), 'modelo']
melhor = cnn_eval if melhor_nome.startswith('CNN') else vgg_eval
melhor.save(MODELS / 'melhor_modelo.keras')
print('Melhor modelo por F1 macro:', melhor_nome)

payload = {
    'aviso_clinico': 'Protótipo acadêmico — NÃO substitui avaliação médica.',
    'seed': SEED,
    'melhor_modelo': melhor_nome,
    'cnn_simples': metricas_cnn,
    'vgg16_tl': metricas_vgg,
}
metricas_path = EVIDENCIAS / 'metricas_teste.json'
metricas_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
print('Métricas salvas em', metricas_path)

# Curvas de treino
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist_cnn.history['accuracy'], label='treino')
axes[0].plot(hist_cnn.history['val_accuracy'], label='val')
axes[0].set_title('CNN simples — acurácia')
axes[0].legend()
axes[1].plot(hist_vgg.history['accuracy'], label='treino (cabeça)')
axes[1].plot(hist_vgg.history['val_accuracy'], label='val (cabeça)')
if 'accuracy' in hist_vgg_ft.history:
    offset = len(hist_vgg.history['accuracy'])
    xs = list(range(offset, offset + len(hist_vgg_ft.history['accuracy'])))
    axes[1].plot(xs, hist_vgg_ft.history['accuracy'], label='treino (fine-tune)')
    axes[1].plot(xs, hist_vgg_ft.history['val_accuracy'], label='val (fine-tune)')
axes[1].set_title('VGG16 TL — acurácia')
axes[1].legend()
plt.tight_layout()
fig.savefig(EVIDENCIAS / 'curvas_treino.png', dpi=140)
plt.show()

## 5. Protótipo Gradio — Assistente Cardiológico Virtual

Envie uma imagem de ECG (laudo JPG). O modelo retorna a classe prevista e as probabilidades.  
**Isto não é diagnóstico.**

In [ ]:
import gradio as gr
from pipeline_imagens import carregar_e_preprocessar

AVISO = (
    '⚠️ AVISO CLÍNICO: Este sistema é um protótipo acadêmico e NÃO substitui avaliação médica. '
)

def prever_ecg(imagem):
    if imagem is None:
        return {c: 0.0 for c in CLASSES}, AVISO
    # Gradio pode entregar PIL ou path/numpy
    if hasattr(imagem, 'save'):
        tmp = EVIDENCIAS / '_tmp_upload.jpg'
        imagem.save(tmp)
        arr = carregar_e_preprocessar(tmp)
    else:
        import cv2
        import numpy as np
        rgb = np.array(imagem)
        if rgb.ndim == 2:
            rgb = np.stack([rgb] * 3, axis=-1)
        rgb = cv2.resize(rgb[..., :3], (224, 224))
        arr = rgb.astype('float32') / 255.0
    batch = np.expand_dims(arr, 0)
    probs = melhor.predict(batch, verbose=0)[0]
    return {CLASSES[i]: float(probs[i]) for i in range(len(CLASSES))}, AVISO

demo = gr.Interface(
    fn=prever_ecg,
    inputs=gr.Image(type='pil', label='Imagem de ECG (laudo)'),
    outputs=[
        gr.Label(num_top_classes=4, label='Classificação simulada'),
        gr.Textbox(label='Aviso clínico', lines=2),
    ],
    title='CardioIA — Assistente Cardiológico Virtual (Fase 4)',
    description=(
        f'Modelo ativo: **{melhor_nome}**. Dataset Mendeley ECG (CC BY 4.0). '
        'Resultados são simulados para fins acadêmicos.'
    ),
    flagging_mode='never',
)

demo.launch(share=EM_COLAB)

## 6. Conclusão

- Duas abordagens treinadas e avaliadas no mesmo split (`seed=42`).
- Métricas e matrizes de confusão em `fase4/docs/evidencias/`.
- Protótipo Gradio apresenta a classificação de forma acessível.

Limitações: laudo completo (possível atalho de template), classes MI/PMI próximas, população da fonte ≠ SUS. Uso estritamente acadêmico.